# imgsz 파라미터 튜닝 (YOLOv8L + MVP 15 클래스)

conf/iou 튜닝 결과에서 선정된 **conf=0.25, conf=0.40 + iou=0.7(고정)**을 기준으로
imgsz(입력 해상도)를 변경하며 최적 조합을 탐색한다.

**실행 전 준비사항:**
1. 런타임 > 런타임 유형 변경 > **GPU (T4)** 선택
2. Google Drive에 `SampleVideo_Scenes` 폴더 확인
   - 경로: `내 드라이브/SampleVideo.Test/SampleVideo_Scenes/`

## 0. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("GPU 확인 완료!")

In [ ]:
# 패키지 설치
!pip install -q ultralytics
print("ultralytics 설치 완료!")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === 경로 설정 ===
DRIVE_SCENE_PATH = "/content/drive/MyDrive/SampleVideo.Test/SampleVideo_Scenes"

import os
scene_count = len([f for f in os.listdir(DRIVE_SCENE_PATH) if f.endswith('.mp4')])
print(f"Scene clips 확인: {scene_count}개")
assert scene_count == 218, f"218개여야 하는데 {scene_count}개 발견됨. 경로를 확인하세요."

## 1. imgsz 튜닝 실험 실행

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from tqdm.auto import tqdm

# =========================
# 기본 설정
# =========================
MODEL_PATH = "yolov8l.pt"
VIDEO_PATH = DRIVE_SCENE_PATH
DATA_YAML = "coco128.yaml"

# MVP 15 class ids
MVP_CLASS_IDS = [16, 26, 39, 41, 45, 53, 56, 57, 59, 60, 62, 63, 65, 67, 72]

# 고정 파라미터 (iou 튜닝 결과)
FIXED_IOU = 0.7

# conf 후보 (conf 튜닝 결과에서 선정)
CONF_CANDIDATES = [0.25, 0.40]

# imgsz 실험 범위
IMGSZ_VALUES = [320, 480, 640, 800, 960, 1280]

total_experiments = len(CONF_CANDIDATES) * len(IMGSZ_VALUES)

# =========================
# 모델 로드
# =========================
print("Loading YOLOv8L model...")
model = YOLO(MODEL_PATH)

scene_files = sorted(Path(VIDEO_PATH).glob("*.mp4"))
print(f"Scene clips: {len(scene_files)}개")
print(f"conf 후보: {CONF_CANDIDATES}")
print(f"iou 고정: {FIXED_IOU}")
print(f"imgsz 실험값: {IMGSZ_VALUES}")
print(f"총 실험 횟수: {total_experiments}회 ({len(CONF_CANDIDATES)} conf x {len(IMGSZ_VALUES)} imgsz)")
print("=" * 55)

In [ ]:
rows = []
exp_num = 0

for conf_val in CONF_CANDIDATES:
    for imgsz_val in IMGSZ_VALUES:
        exp_num += 1
        exp_id = f"S{exp_num}"

        print(f"\n{'='*55}")
        print(f"  [{exp_num}/{total_experiments}] conf={conf_val}, iou={FIXED_IOU}, imgsz={imgsz_val}")
        print(f"{'='*55}")

        # --- COCO128 Validation ---
        print("  [1/2] COCO128 Validation...")
        val_start = time.time()
        metrics = model.val(
            data=DATA_YAML,
            imgsz=imgsz_val,
            conf=conf_val,
            iou=FIXED_IOU,
            classes=MVP_CLASS_IDS,
            verbose=False
        )
        val_time = time.time() - val_start

        precision_arr = metrics.box.p
        recall_arr = metrics.box.r
        precision = float(np.nanmean(precision_arr)) if hasattr(precision_arr, "__len__") else float(precision_arr)
        recall = float(np.nanmean(recall_arr)) if hasattr(recall_arr, "__len__") else float(recall_arr)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"  Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f} ({val_time:.1f}s)")

        # --- SampleVideo_Scenes 추론 (stream=True) ---
        print(f"  [2/2] SampleVideo_Scenes 추론 ({len(scene_files)} clips)...")
        total_detected = 0
        start_time = time.time()

        pbar = tqdm(scene_files, desc=f"  conf={conf_val} imgsz={imgsz_val}", leave=True)
        for scene_file in pbar:
            results = model.predict(
                source=str(scene_file),
                conf=conf_val,
                iou=FIXED_IOU,
                imgsz=imgsz_val,
                classes=MVP_CLASS_IDS,
                stream=True,
                verbose=False
            )
            for result in results:
                if result.boxes is not None:
                    total_detected += len(result.boxes)
            pbar.set_postfix({"detected": total_detected})

        elapsed = time.time() - start_time
        det_per_sec = total_detected / elapsed if elapsed > 0 else 0

        print(f"  => 탐지: {total_detected}개 | 시간: {elapsed:.1f}s | {det_per_sec:.1f} det/s")

        rows.append({
            "실험": exp_id,
            "conf": conf_val,
            "iou": FIXED_IOU,
            "imgsz": imgsz_val,
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "F1": round(f1, 4),
            "탐지 수": total_detected,
            "소요 시간(s)": round(elapsed, 1),
            "탐지/sec": round(det_per_sec, 1)
        })

df = pd.DataFrame(rows)
print("\n" + "=" * 55)
print("  모든 실험 완료!")
print("=" * 55)

## 2. 결과 확인

In [ ]:
from IPython.display import display

# F1 최고값 강조
best_idx = df["F1"].idxmax()
best_row = df.loc[best_idx]

def highlight_best(row):
    if row.name == best_idx:
        return ['background-color: #90EE90; font-weight: bold'] * len(row)
    return [''] * len(row)

print("=== conf=0.25 그룹 ===")
display(df[df['conf'] == 0.25].style.apply(highlight_best, axis=1).format({
    'Precision': '{:.4f}', 'Recall': '{:.4f}', 'F1': '{:.4f}'
}))

print("\n=== conf=0.40 그룹 ===")
display(df[df['conf'] == 0.40].style.apply(highlight_best, axis=1).format({
    'Precision': '{:.4f}', 'Recall': '{:.4f}', 'F1': '{:.4f}'
}))

print(f"\n★ Best F1 = {best_row['F1']} @ conf={best_row['conf']}, iou={best_row['iou']}, imgsz={best_row['imgsz']}")
print(f"  Precision={best_row['Precision']}, Recall={best_row['Recall']}")
print(f"  탐지 수={best_row['탐지 수']}, 소요 시간={best_row['소요 시간(s)']}s")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {0.25: 'royalblue', 0.40: 'tomato'}
markers = {0.25: 'o', 0.40: 's'}

# --- 왼쪽: F1 비교 ---
for conf_val in CONF_CANDIDATES:
    subset = df[df['conf'] == conf_val]
    axes[0].plot(subset['imgsz'], subset['F1'], f'-{markers[conf_val]}',
                color=colors[conf_val], label=f'conf={conf_val}', linewidth=2, markersize=8)

axes[0].axvline(x=best_row['imgsz'], color='green', linestyle='--', alpha=0.4)
axes[0].set_xlabel('Image Size (imgsz)', fontsize=12)
axes[0].set_ylabel('F1 Score', fontsize=12)
axes[0].set_title('F1 Score vs imgsz', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- 가운데: Precision vs Recall ---
for conf_val in CONF_CANDIDATES:
    subset = df[df['conf'] == conf_val]
    axes[1].plot(subset['imgsz'], subset['Precision'], f'--{markers[conf_val]}',
                color=colors[conf_val], label=f'P (conf={conf_val})', linewidth=1.5, alpha=0.7)
    axes[1].plot(subset['imgsz'], subset['Recall'], f'-{markers[conf_val]}',
                color=colors[conf_val], label=f'R (conf={conf_val})', linewidth=2)

axes[1].set_xlabel('Image Size (imgsz)', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Precision & Recall vs imgsz', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# --- 오른쪽: 소요 시간 vs 탐지 수 (트레이드오프) ---
for conf_val in CONF_CANDIDATES:
    subset = df[df['conf'] == conf_val]
    axes[2].plot(subset['소요 시간(s)'], subset['F1'], f'-{markers[conf_val]}',
                color=colors[conf_val], label=f'conf={conf_val}', linewidth=2, markersize=8)
    for _, row in subset.iterrows():
        axes[2].annotate(f"{int(row['imgsz'])}",
                        (row['소요 시간(s)'], row['F1']),
                        textcoords="offset points", xytext=(5, 5), fontsize=8)

axes[2].set_xlabel('Processing Time (s)', fontsize=12)
axes[2].set_ylabel('F1 Score', fontsize=12)
axes[2].set_title('F1 vs Speed Trade-off', fontsize=13)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('imgsz_tuning_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장: imgsz_tuning_result.png")

## 3. 결과 저장 & 다운로드

In [ ]:
# CSV 저장
csv_path = "tune_imgsz_results.csv"
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"CSV 저장: {csv_path}")

# Drive에도 백업
import shutil
drive_save_dir = "/content/drive/MyDrive/SampleVideo.Test"
df.to_csv(f"{drive_save_dir}/tune_imgsz_results.csv", index=False, encoding='utf-8-sig')
shutil.copy('imgsz_tuning_result.png', f'{drive_save_dir}/imgsz_tuning_result.png')
print(f"Drive 백업 완료")

# 다운로드
from google.colab import files
files.download(csv_path)
files.download('imgsz_tuning_result.png')
print("\n다운로드 시작!")

In [ ]:
# Report용 마크다운 테이블 출력
print("=" * 60)
print("  아래 내용을 Project_Report.md에 붙여넣으세요")
print("=" * 60)
print()
print(df.to_markdown(index=False))
print()
print(f"\n★ Best F1 = {best_row['F1']} @ conf={best_row['conf']}, iou={best_row['iou']}, imgsz={best_row['imgsz']}")
print(f"\n=== 최종 파라미터 후보 ===")
print(f"  conf={best_row['conf']}, iou={best_row['iou']}, imgsz={int(best_row['imgsz'])}")